# Project: Drowsiness Detection

## Install Packages

In [1]:
!pip3 install torch torchvision

  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
Using cached setuptools-81.0.0-py3-none-any.whl (1.1 MB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 82.0.1
    Uninstalling setuptools-82.0.1:
      Successfully uninstalled setuptools-82.0.1

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
!pip3 install ultralytics labelme numpy albumentations matplotlib


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Import YOLO26

In [3]:
from ultralytics import YOLO
import os
import json
import uuid
import numpy as np
import matplotlib.pyplot as plt

In [4]:
dataset_folder = 'dataset'

In [5]:
model = YOLO("yolo26n.pt")

In [6]:
img = os.path.join('test', 'test1.jpg')
results = model(img)
print(results)


image 1/1 /Users/rkg/Desktop/Project/test/test1.jpg: 640x640 1 person, 1 chair, 49.7ms
Speed: 3.4ms preprocess, 49.7ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 640)
[ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
keypoints: None
masks: None
names: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl',

## Dataset Pre-Processing 

In [ ]:
normal_path = os.path.join(dataset_folder, 'normal')
drowsy_path = os.path.join(dataset_folder, 'drowsy')
images_path = os.path.join(dataset_folder, 'images')

normal_images = os.listdir(normal_path)
drowsy_images = os.listdir(drowsy_path)

for normal_image in normal_images:
    print(os.path.join(normal_path, normal_image), os.path.join(images_path, 'normal.' + str(uuid.uuid1()) + '.png'))
    os.replace(os.path.join(normal_path, normal_image), os.path.join(images_path, 'normal.' + str(uuid.uuid1()) + '.png'))
    
for drowsy_image in drowsy_images:
    print(os.path.join(drowsy_path, drowsy_image), os.path.join(images_path, 'drowsy.' + str(uuid.uuid1()) + '.png'))
    os.replace(os.path.join(drowsy_path, drowsy_image), os.path.join(images_path, 'drowsy.' + str(uuid.uuid1()) + '.png'))

## Labeling

In [7]:
!labelme

2026-04-15 11:14:13.932 | INFO     | labelme.__main__:main:226 - Starting Labelme 6.0.0
qt.qpa.fonts: Populating font family aliases took 67 ms. Replace uses of missing font family "Monospace" with one that exists to avoid this cost. 
2026-04-15 11:14:25.647 | DEBUG    | labelme.app:_open_next_image:2196 - there is no next image


Convert labelme coordinates to YOLO coordinates

In [8]:
def labelme_to_yolo(data):
    if data['shapes'][0]['label'] == "normal":
        label_class = 0
    else:
        label_class = 1

    image_width = data['imageWidth']
    image_height = data['imageHeight']
    
    x_min = data['shapes'][0]['points'][0][0]
    x_max = data['shapes'][0]['points'][1][0]
    y_min = data['shapes'][0]['points'][0][1]
    y_max = data['shapes'][0]['points'][1][1]

    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2

    bbox_width = x_max - x_min
    bbox_height = y_max - y_min
    
    return f'{label_class} {x_center/image_width} {y_center/image_height} {bbox_width/image_width} {bbox_height/image_height}'
    
    
    #print(x_min, x_max, y_min, y_max)
    #print(data['imageWidth'])
    #print(data['imageHeight'])

In [9]:
labels = os.listdir(os.path.join("dataset", "labels_labelme"))

for label in labels:
    with open(os.path.join("dataset", "labels_labelme", label), 'r', encoding='utf-8') as label_file:
        data = json.load(label_file)
        output = labelme_to_yolo(data)
    
    with open(os.path.join("dataset", "labels", f'{label.split('.json')[0]}.txt'), 'w', encoding='utf-8') as f:
        f.write(output)

    

## Augmentation

In [10]:
import albumentations as alb

In [11]:
train_transform = alb.Compose([alb.HorizontalFlip(p=0.5),
                               alb.RGBShift(p=0.2),
                               alb.RandomBrightnessContrast(p=0.2),
                               alb.VerticalFlip(p=0.5),
                               alb.RandomGamma(p=0.2)
                              ], bbox_params=alb.BboxParams(format='yolo',
                              label_fields=['class_labels']
                             ))

In [12]:
total_aug = 20

for image in os.listdir(os.path.join(dataset_folder, 'images')):
    img = cv2.imread(os.path.join(dataset_folder, 'images', image))
    
    label_filepath = os.path.join(dataset_folder, 'labels', image.split(".png")[0] + '.txt')
    with open(label_filepath, 'r') as f:
        content = f.read()
    class_name = content.split(" ")[0]
    img_coords = np.array([content.split(" ")[1], content.split(" ")[2], content.split(" ")[3], content.split(" ")[4].split("\n")[0]], dtype=np.float32)

    try:
        for i in range(total_aug):
            aug_img = train_transform(image=img, bboxes=[img_coords], class_labels=[class_name])

            cv2.imwrite(os.path.join(dataset_folder, 'images', f'{image.split(".png")[0]}.{i}.jpg'), aug_img['image'])

            aug_content = f'{class_name} {aug_img["bboxes"][0][0]} {aug_img["bboxes"][0][1]} {aug_img["bboxes"][0][2]} {aug_img["bboxes"][0][3]}'
            with open(os.path.join(dataset_folder, 'labels', f'{image.split(".png")[0]}.{i}.txt'), 'w') as file:
                file.write(aug_content)

    except Exception as e:
        print(e)

NameError: name 'cv2' is not defined

## Training Model

In [ ]:
results = model.train(data="dataset.yaml", epochs=100, imgsz=640)

## Prediction

After total 7 experiments, most of them trained for 100 ephocs

In [13]:
model = YOLO(os.path.join("runs", "detect", "train7", "weights", "last.pt"))

In [14]:
results = model.predict(os.path.join('test', 'test1.jpg'))
results[0].show()


image 1/1 /Users/rkg/Desktop/Project/test/test1.jpg: 640x640 1 drowsy, 72.6ms
Speed: 8.1ms preprocess, 72.6ms inference, 3.1ms postprocess per image at shape (1, 3, 640, 640)


In [15]:
results = model.predict(os.path.join('test', 'test2.jpg'))
results[0].show()


image 1/1 /Users/rkg/Desktop/Project/test/test2.jpg: 448x640 1 drowsy, 103.0ms
Speed: 26.5ms preprocess, 103.0ms inference, 3.2ms postprocess per image at shape (1, 3, 448, 640)


In [16]:
results = model.predict(os.path.join('test', 'test3.jpg'))
results[0].show()


image 1/1 /Users/rkg/Desktop/Project/test/test3.jpg: 448x640 1 normal, 48.4ms
Speed: 3.3ms preprocess, 48.4ms inference, 0.2ms postprocess per image at shape (1, 3, 448, 640)


In [17]:
results = model.predict(os.path.join('test', 'test4.jpg'))
results[0].show()


image 1/1 /Users/rkg/Desktop/Project/test/test4.jpg: 384x640 1 normal, 41.3ms
Speed: 3.7ms preprocess, 41.3ms inference, 0.1ms postprocess per image at shape (1, 3, 384, 640)


In [18]:
results = model.predict(os.path.join('test', 'test5.jpg'))
results[0].show()


image 1/1 /Users/rkg/Desktop/Project/test/test5.jpg: 384x640 1 drowsy, 45.4ms
Speed: 3.2ms preprocess, 45.4ms inference, 0.2ms postprocess per image at shape (1, 3, 384, 640)
